In [ ]:
import awkward as ak
import dask
import hist
import hist.dask
import json
from coffea import processor
from coffea.nanoevents import BaseSchema, NanoAODSchema 
import corrections
import matplotlib.pyplot as plt


class MyZPeak(processor.ProcessorABC):
    def __init__(self, mode="virtual"):
        assert mode in ["eager", "virtual", "dask"]
        self._mode = mode
        
    def process(self, events):
        dataset = events.metadata['dataset']
        isRealData = "genWeight" not in events.fields
        sumw = 0. if isRealData else ak.sum(events.genWeight, axis=0)
        cutflow = {"start": ak.num(events, axis=0)}
        
        if isRealData:
            events = events[
                corrections.lumimask(events.run, events.luminosityBlock)
            ]
            cutflow["lumimask"] = ak.num(events, axis=0)
    
        events["goodmuons"] = events.Muon[
            (events.Muon.pt >= 20.)
            & events.Muon.tightId
        ]

        events = events[
            (ak.num(events.goodmuons) == 2)
            & (ak.sum(events.goodmuons.charge, axis=1) == 0)
        ]
        cutflow["ossf"] = ak.num(events, axis=0)
        
        # add first and second muon p4 in every event together
        events["zcand"] = events.goodmuons[:, 0] + events.goodmuons[:, 1]

        # require trigger
        events = events[
            # https://twiki.cern.ch/twiki/bin/view/CMS/MuonHLT2018
            events.HLT.Mu17_TrkIsoVVL_Mu8_TrkIsoVVL_DZ_Mass3p8
        ]
        weight = 1 * ak.ones_like(events.event) if isRealData else events.genWeight
        cutflow["trigger"] = ak.num(events, axis=0)

        if self._mode == "dask":
            hist_class = hist.dask.Hist
        else:
            hist_class = hist.Hist

        h = hist_class.new.Reg(120, 0., 120., label=r"$m_{\mu\mu}$ [GeV]").Weight()

        if self._mode == "dask":
            return {
                    "entries": ak.num(events, axis=0),
                    "sumw": sumw,
                    "cutflow": cutflow,
                    "mass": h.fill(events.zcand.mass, weight=weight)
                }
        else:
            return {
                dataset: {
                    "entries": ak.num(events, axis=0),
                    "sumw": sumw,
                    "cutflow": cutflow,
                    "mass": h.fill(events.zcand.mass, weight=weight)
                }
            }

    def postprocess(self, accumulator):
        return accumulator

In [ ]:
from dask.distributed import Client

client = Client("tls://localhost:8786")
client

In [ ]:
import shutil
shutil.make_archive("corrections", "zip", base_dir="corrections")

In [ ]:
client.upload_file("corrections.zip")

In [ ]:
with open("fileset.json", "rt") as file:
    initial_fileset = json.load(file)

# Scaling in Virtual mode

In [ ]:
run = processor.Runner(
    executor = processor.DaskExecutor(client=client, compression=None),
    schema=NanoAODSchema,
    chunksize=100_000,
    skipbadfiles=True,
    savemetrics=True,
    maxchunks=7,
)

small_result, small_report = run(
    initial_fileset,
    processor_instance=MyZPeak("virtual"),
)